# 3. Post-Submission Review

复盘 PnL、找偏差原因、对比预期 vs 实际。

In [ ]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices_wide, load_trades
from utils.orderbook import compute_wall_mid
import polars as pl
import numpy as np
import plotly.graph_objects as go

## 加载提交结果

将官网下载的结果 CSV 放入 `data/submissions/` 目录。

In [ ]:
# submission_path = 'data/submissions/round1_attempt1.csv'
# sub = pl.read_csv(submission_path, separator=';')
# sub.head()

## PnL 曲线 vs 产品

In [ ]:
# 从 prices 文件的 profit_and_loss 列提取 PnL 曲线
ROUND = 0
DAY = -1

pw = load_prices_wide(ROUND, DAY)

fig = go.Figure()
for product in pw['product'].unique().to_list():
    pp = pw.filter(pl.col('product') == product).sort('timestamp')
    fig.add_trace(go.Scatter(
        x=pp['timestamp'].to_list(),
        y=pp['profit_and_loss'].to_list(),
        mode='lines',
        name=product,
    ))

fig.update_layout(title=f'PnL Day {DAY}', height=400)
fig.show()

## 偏差分析

- 哪个时段亏钱？
- 是 fair 估计偏了还是 fill 不如预期？
- 跨天稳定性如何？

In [ ]:
# 对比两天的 PnL
for day in [-2, -1]:
    pw = load_prices_wide(ROUND, day)
    for product in pw['product'].unique().to_list():
        pp = pw.filter(pl.col('product') == product)
        final_pnl = pp.sort('timestamp')['profit_and_loss'].to_list()[-1]
        print(f'Day {day}, {product}: final PnL = {final_pnl:.1f}')